# Stage 6 — Balanced fuzzy rule evaluation

In [1]:
from pathlib import Path
import os, warnings
import pandas as pd
import numpy as np
warnings.filterwarnings("ignore")

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
except Exception:
    pass

def locate_root():
    candidates = [
        Path(os.environ.get("MFAR_PROJECT_ROOT","")),
        Path("/content/drive/MyDrive/MFAR_Modular_Colab_Pipeline"),
        Path.cwd(), Path.cwd().parent
    ]
    for p in candidates:
        if str(p) and (p/"stage_output").exists() and (p/"config").exists():
            return p.resolve()
    raise FileNotFoundError("Project root not found.")

ROOT=locate_root()
RAW=ROOT/"data_raw"
CFG=ROOT/"config"
STAGE=ROOT/"stage_output"
for i in range(1,8):
    (STAGE/f"stage_{i:02d}").mkdir(parents=True,exist_ok=True)
print("ROOT =",ROOT)

Mounted at /content/drive
ROOT = /content/drive/MyDrive/MFAR_Modular_Colab_Pipeline


In [2]:
df=pd.read_csv(STAGE/"stage_05"/"05_fuzzy_memberships.csv")
def FAND(*v): return float(min(float(x) for x in v))
def FOR(*v): return float(max(float(x) for x in v))

RULE_TEXT={
"R01":"IF origin queue LOW AND berth AVAILABLE THEN NO_INTERVENTION",
"R02":"IF destination queue LOW AND berth AVAILABLE THEN MAINTAIN_SPEED",
"R03":"IF origin queue MEDIUM OR HIGH AND berth AVAILABLE THEN DEPART_NOW",
"R04":"IF berth UNAVAILABLE AND wait MEDIUM OR LONG THEN HOLD_DEPARTURE",
"R05":"IF berth UNAVAILABLE AND wait MEDIUM OR LONG THEN REDUCE_SPEED",
"R06":"IF origin queue HIGH AND service gap MEDIUM OR LONG THEN INCREASE_SERVICE_PRIORITY",
"R07":"IF destination queue HIGH OR CRITICAL AND wait MEDIUM OR LONG THEN RESCHEDULE_HEADWAY",
"R08":"IF origin queue CRITICAL AND service gap LONG AND capacity shortfall HIGH THEN ADD_VESSEL",
"R09":"IF confidence LOW AND operational risk exists THEN ALERT_OPERATOR",
"R10":"IF origin queue CRITICAL AND berth UNAVAILABLE THEN ALERT_OPERATOR"
}

In [3]:
def infer(r):
    rules={
     "R01":(FAND(r.mu_origin_queue_low,r.mu_berth_available),"NO_INTERVENTION"),
     "R02":(FAND(r.mu_destination_queue_low,r.mu_berth_available),"MAINTAIN_SPEED"),
     "R03":(FAND(FOR(r.mu_origin_queue_medium,r.mu_origin_queue_high),
                  r.mu_berth_available),"DEPART_NOW"),
     "R04":(FAND(r.mu_berth_unavailable,FOR(r.mu_wait_medium,r.mu_wait_long)),
            "HOLD_DEPARTURE"),
     "R05":(FAND(r.mu_berth_unavailable,FOR(r.mu_wait_medium,r.mu_wait_long)),
            "REDUCE_SPEED"),
     "R06":(FAND(r.mu_origin_queue_high,
                  FOR(r.mu_service_gap_medium,r.mu_service_gap_long)),
            "INCREASE_SERVICE_PRIORITY"),
     "R07":(FAND(FOR(r.mu_destination_queue_high,r.mu_destination_queue_critical),
                  FOR(r.mu_wait_medium,r.mu_wait_long)),
            "RESCHEDULE_HEADWAY"),
     "R08":(FAND(r.mu_origin_queue_critical,r.mu_service_gap_long,
                  r.mu_capacity_shortfall_high),"ADD_VESSEL"),
     "R09":(FAND(r.mu_confidence_low,
                  FOR(r.mu_origin_queue_high,r.mu_origin_queue_critical,
                      r.mu_berth_unavailable)),"ALERT_OPERATOR"),
     "R10":(FAND(r.mu_origin_queue_critical,r.mu_berth_unavailable),
            "ALERT_OPERATOR")
    }
    actions={}
    for rid,(s,a) in rules.items():
        actions[a]=max(actions.get(a,0.0),s)
    if max(actions.values() or [0])==0:
        actions["NO_INTERVENTION"]=1.0
    sel=max(actions,key=actions.get)
    ss=actions[sel]
    dominant="|".join(rid for rid,(s,a) in rules.items()
                      if a==sel and abs(s-ss)<1e-12)
    return pd.Series({
      **{f"firing_{rid}":s for rid,(s,a) in rules.items()},
      "selected_action":sel,"selected_rule_strength":ss,
      "dominant_rule":dominant
    })

out=pd.concat([df,df.apply(infer,axis=1)],axis=1)
S6=STAGE/"stage_06"
out.to_csv(S6/"06_rule_evaluation.csv",index=False)
pd.DataFrame({"rule_id":list(RULE_TEXT),"rule_text":list(RULE_TEXT.values())}).to_csv(
 S6/"06_fuzzy_rule_catalog.csv",index=False)
pd.DataFrame([
 {"rule_id":rid,"active_rows":int((out[f"firing_{rid}"]>0).sum()),
  "mean_firing_strength":float(out[f"firing_{rid}"].mean()),
  "max_firing_strength":float(out[f"firing_{rid}"].max())}
 for rid in RULE_TEXT
]).to_csv(S6/"06_rule_firing_summary.csv",index=False)
display(out["selected_action"].value_counts())

,count
selected_action,
NO_INTERVENTION,10
DEPART_NOW,10
HOLD_DEPARTURE,2
MAINTAIN_SPEED,2
ADD_VESSEL,2
ALERT_OPERATOR,1
INCREASE_SERVICE_PRIORITY,1
